In [1]:
from pltconfig_tex import *

In [2]:
from pandas import read_excel, read_csv

In [3]:
supply_df = read_csv("../../../gac_sol_1.0mw_supply-curve-aggregation.csv")

In [4]:
P2E_ratio = 0.25

In [5]:
num_sites = supply_df.shape[0]
site_idxs = arange(num_sites) + 1
carbon_reduction_optimal = zeros((num_sites, 3, 2))
carbon_subtracted_optimal = zeros((num_sites, 3, 2))
tot_batt_throughput_optimal = zeros((num_sites, 3, 2))
tot_batt_dchg_optimal = zeros((num_sites, 3, 2))
tot_solar_to_batt_optimal = zeros((num_sites, 3, 2))
tot_solar = zeros((num_sites, 3, 2))
tot_solar_useful = zeros((num_sites, 3, 2))
tot_solar_curtailed_optimal = zeros((num_sites, 3, 2))
fractions_optimal = zeros((num_sites, 3, 2, 3))
fractions_batt_charge_optimal = zeros((num_sites, 3, 2, 2))
cycles_optimal = zeros((num_sites, 3, 2))
    
nom_solar_cap = supply_df["capacity"].to_numpy()

for j, shift in enumerate([-4, 0, 4]):
    for k, batt_size in enumerate([0, 100]):

        summary_optimal = read_excel(f"./summary_shift_{shift:.0f}_mu_0.85_P2Eratio_{P2E_ratio:.2f}_outmat.xlsx", sheet_name=f"{batt_size:.0f} MWh Battery - Optimal")
        
        carbon_reduction_optimal[:, j, k] = summary_optimal["CO2_removed_pcent"].to_numpy()
        
        carbon_subtracted_optimal[:, j, k] = summary_optimal["CO2_removed_t"].to_numpy()

        tot_solar[:, j, k] = summary_optimal["tot_solar_MWh"].to_numpy()
        tot_solar_useful[:, j, k] = (summary_optimal["tot_solar_MWh"] - summary_optimal["tot_solar_curtailed_MWh"]).to_numpy()

        tot_solar_to_batt_optimal[:, j, k] = summary_optimal["tot_solar_to_batt_MWh"].to_numpy()

        tot_solar_curtailed_optimal[:, j, k] = summary_optimal["tot_solar_curtailed_MWh"].to_numpy()

        tot_batt_dchg_optimal[:, j, k] = summary_optimal["tot_batt_dchg_MWh"].to_numpy()

        tot_batt_throughput_optimal[:, j, k] = (summary_optimal["tot_batt_dchg_MWh"] + summary_optimal["tot_batt_chg_MWh"]).to_numpy()

        fractions_optimal[:, j, k, 0] = summary_optimal["solar_fraction_num"].to_numpy()
        fractions_optimal[:, j, k, 1] = summary_optimal["ESS_fraction_num"].to_numpy()
        fractions_optimal[:, j, k, 2] = summary_optimal["grid_fraction_num"].to_numpy()

        if batt_size > 0.0:
            fractions_batt_charge_optimal[:, j, k, 0] = summary_optimal["solar_chargeFraction_num"].to_numpy()
            fractions_batt_charge_optimal[:, j, k, 1] = summary_optimal["grid_chargeFraction_num"].to_numpy()

        cycles_optimal[:, j, k] = summary_optimal["ESS_cycles_num"].to_numpy()

In [6]:
savez(f"processed_results_siteVariation", 
      carbon_reduction_optimal=carbon_reduction_optimal,
      carbon_subtracted_optimal=carbon_subtracted_optimal, 
      tot_solar=tot_solar,
      tot_solar_curtailed_optimal=tot_solar_curtailed_optimal,
      tot_batt_dchg_optimal=tot_batt_dchg_optimal,
      tot_batt_throughput_optimal=tot_batt_throughput_optimal, 
      fractions_optimal=fractions_optimal, 
      fractions_batt_charge_optimal=fractions_batt_charge_optimal, 
      cycles_optimal=cycles_optimal
     )